# Exploración de features

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

from scripts.data.load_data import cargar_raw
from scripts.data.clean_data import limpiar_datos

df = cargar_raw("titanic_dataset.csv")

df_limpio = limpiar_datos(df)

df_exploratorio = df_limpio.copy()

In [2]:
df_exploratorio.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S


In [3]:
import pandas as pd
import numpy as np

# modelado
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# métricas
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# encoding
from sklearn.preprocessing import OneHotEncoder


In [4]:
def evaluar_features(df, features):
    
    X = df[features]
    y = df["Survived"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)
    
    return {
        "features": features,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }

## SibSp y Parch

In [5]:
evaluar_features(df_exploratorio, ["SibSp", "Parch"])

{'features': ['SibSp', 'Parch'],
 'accuracy': 0.5921787709497207,
 'f1': 0.16091954022988506}

In [6]:
df_exploratorio['FamilySize'] = df_exploratorio['SibSp'] + df_exploratorio['Parch'] + 1

In [8]:
evaluar_features(df_exploratorio, ["FamilySize"])

{'features': ['FamilySize'], 'accuracy': 0.5865921787709497, 'f1': 0.0}

In [11]:
evaluar_features(df_exploratorio, ["FamilySize", "Age", "Fare"])

{'features': ['FamilySize', 'Age', 'Fare'],
 'accuracy': 0.6536312849162011,
 'f1': 0.3541666666666667}

In [12]:
evaluar_features(df_exploratorio, ["SibSp","Parch", "Age", "Fare"])

{'features': ['SibSp', 'Parch', 'Age', 'Fare'],
 'accuracy': 0.6815642458100558,
 'f1': 0.44660194174757284}

In [34]:
def FamilySize_agrupado (size):
    if size == 1:
        return "Solo"
    elif 2 <= size <= 4:
        return "Pequeña"
    else:
        return "Grande"

In [35]:
df_exploratorio['FamilySize_agrupado'] = df_exploratorio['FamilySize'].apply(FamilySize_agrupado)
df_exploratorio["FamilySize_agrupado"] = pd.Categorical(df_exploratorio["FamilySize_agrupado"], categories=["Solo", "Pequeña", "Grande"])

In [36]:
df_exploratorio = pd.get_dummies( df_exploratorio, columns=["FamilySize_agrupado"], drop_first=True, dtype=int)

In [37]:
df_exploratorio.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,FamilySize_agrupado_Pequeña,FamilySize_agrupado_Grande
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S,2,1,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C,2,1,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S,1,0,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S,2,1,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S,1,0,0


In [44]:
evaluar_features(df_exploratorio, ["FamilySize_agrupado_Pequeña", "FamilySize_agrupado_Grande", "Age", "Fare"])

{'features': ['FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age',
  'Fare'],
 'accuracy': 0.7150837988826816,
 'f1': 0.5641025641025641}

In [39]:
evaluar_features(df_exploratorio, ['FamilySize_agrupado_Pequeña', 'FamilySize_agrupado_Grande'])

{'features': ['FamilySize_agrupado_Pequeña', 'FamilySize_agrupado_Grande'],
 'accuracy': 0.6983240223463687,
 'f1': 0.5970149253731343}

## Name

In [41]:
df_exploratorio['Name'].shape

(891,)

In [47]:
df_exploratorio["Title"] = df_exploratorio["Name"].str.extract(r",\s*([^\.]+)\.")
df_exploratorio["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Mlle              2
Major             2
Col               2
the Countess      1
Capt              1
Ms                1
Sir               1
Lady              1
Mme               1
Don               1
Jonkheer          1
Name: count, dtype: int64

In [48]:
def Title_agrupado(t):
    if t in ["Mr"]:
        return "Hombre"
    elif t in ["Mrs", "Miss"]:
        return "Mujer"
    elif t in ["Master"]:
        return "Niño"
    else:
        return "Rare"

In [49]:
df_exploratorio["Title_agrupado"] = df_exploratorio["Title"].apply(Title_agrupado)

In [51]:
df_exploratorio["Title_agrupado"] = pd.Categorical(
    df_exploratorio["Title_agrupado"],
    categories=["Hombre", "Mujer", "Niño", "Rare"]
)

In [53]:
df_exploratorio = pd.get_dummies(
    df_exploratorio,
    columns=["Title_agrupado"],
    drop_first=True, 
    dtype=int
)

In [54]:
df_exploratorio.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,FamilySize_agrupado_Pequeña,FamilySize_agrupado_Grande,Title,Title_agrupado_Mujer,Title_agrupado_Niño,Title_agrupado_Rare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S,2,1,0,Mr,0,0,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C,2,1,0,Mrs,1,0,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S,1,0,0,Miss,1,0,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S,2,1,0,Mrs,1,0,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S,1,0,0,Mr,0,0,0


In [55]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", "Title_agrupado_Mujer", "Title_agrupado_Niño"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño'],
 'accuracy': 0.776536312849162,
 'f1': 0.7183098591549296}

In [56]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", "Title_agrupado_Mujer", "Title_agrupado_Niño", "Age", "Fare"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Age',
  'Fare'],
 'accuracy': 0.776536312849162,
 'f1': 0.7222222222222223}

In [57]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño", 
                                    "Age", 
                                    "Fare", 
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande" ])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Age',
  'Fare',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'accuracy': 0.8156424581005587,
 'f1': 0.7692307692307693}